> ## SOLUTION / ANSWER KEY &mdash; Lab 6.1
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-01-your-first-tool.ipynb`](../lab-01-your-first-tool.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.1 &mdash; Your First Tool with @tool

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Wrap a function as a tool with LangChain's @tool decorator
- See the name, description (from the docstring) and args the model reads
- Call the tool with .invoke() and get a real result

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; Defining a tool](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-01"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
In Module 5 you built tools by hand as dicts. A framework does it for you: LangChain's **`@tool`**
decorator (from **`langchain_core.tools`**) turns a plain function into a **`StructuredTool`** with a
**name**, a **description** (taken from the **docstring** &mdash; the text the model reads to decide
when to use it), an **args** schema, and an **`.invoke()`** method. Same idea as before &mdash; one
decorator instead of a dict.

In [ ]:
from langchain_core.tools import tool

@tool
def greet(name: str) -> str:
    """Say hello to someone by name."""
    return "Hello, " + name + "!"

print("name:", greet.name, "| description:", greet.description)
print("args:", list(greet.args))
print("greet.invoke('Ada') ->", greet.invoke("Ada"))

## Your Turn
Write two real tools &mdash; a **calculator** and a **lookup** &mdash; each with a clear docstring
(the description the model reads), then call them with `.invoke()`.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

from langchain_core.tools import tool

@tool
def calculator(expression: str) -> float:
    """Do exact arithmetic on an expression such as 2+2 or 120000/2."""
    return safe_calc(expression)

@tool
def lookup(key: str) -> str:
    """Look up a known fact by its key, for example capital of france."""
    facts = {"capital of france": "Paris", "population of metropolis": "120000"}
    return facts.get(key.lower().strip(), "unknown")

try:
    print('calculator.name :', calculator.name)
    print('calculator.args :', list(calculator.args))
    print('calculator.invoke("120000/2") =', calculator.invoke('120000/2'))
    print('lookup.invoke("capital of france") =', lookup.invoke('capital of france'))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("calculator is a Tool named 'calculator'", lambda: calculator.name == "calculator")
expect_true("its args schema is ['expression']", lambda: list(calculator.args) == ["expression"])
expect_true("the description comes from the docstring", lambda: "arithmetic" in calculator.description.lower())
expect_true("calculator.invoke computes 120000/2 == 60000.0", lambda: calculator.invoke("120000/2") == 60000.0)
expect_true("lookup finds a known fact", lambda: lookup.invoke("capital of france") == "Paris")
expect_true("lookup returns 'unknown' for a missing key", lambda: lookup.invoke("price of gold") == "unknown")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
`@tool` turns a function into a real LangChain Tool the agent can call. The docstring is the description the model reads -- next we see why that description is the tool's real API.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>